In [2]:
import vllm
import torch

print("vLLM:", vllm.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

vLLM: 0.32.0
PyTorch: 2.13.0+cu129
CUDA available: True
GPU: Tesla T4


In [3]:
!pkill -f VLLM::EngineCore  #to clear GPU memory

In [7]:
!vllm serve Qwen/Qwen2.5-0.5B-Instruct \
    --host 127.0.0.1 \
    --port 8000 \
    --max-model-len 1024 \
    --gpu-memory-utilization 0.80

(APIServer pid=344) INFO 10-20 05:04:43 [api_utils.py:347] 
(APIServer pid=344) INFO 10-20 05:04:43 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=344) INFO 10-20 05:04:43 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.31.0
(APIServer pid=344) INFO 10-20 05:04:43 [api_utils.py:347]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-0.5B-Instruct
(APIServer pid=344) INFO 10-20 05:04:43 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=344) INFO 10-20 05:04:43 [api_utils.py:347] 
(APIServer pid=344) INFO 10-20 05:04:43 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen2.5-0.5B-Instruct', 'host': '127.0.0.1', 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'max_model_len': 1024, 'gpu_memory_utilization': 0.8}
config.json: 100%|█████████████████████████████| 659/659 [00:00<00:00, 2.84MB/s]
(APIServer pid=344) INFO 10-20 05:05:00 [model.py:699] Resolved architecture: Qwen2ForCausalLM
(APIServer pid=344) WARNING 10-20 05:05:00 [model.py:2389] Your device 'T

In [14]:
import subprocess
import time
import requests

with open("/content/vllm-server.log", "w") as log:
    process = subprocess.Popen(
        [
            "vllm", "serve", "Qwen/Qwen2.5-0.5B-Instruct",
            "--host", "127.0.0.1",
            "--port", "8000",
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

print("Server PID:", process.pid)

Server PID: 774


In [15]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",
    api_key="EMPTY"
)

response = client.chat.completions.create(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    messages=[
        {"role": "user", "content": "Explain RAG in simple words."}
    ],
    max_tokens=100,
    temperature=0.9,
    stream=False
)

print(response.choices[0].message.content)

RAG stands for Research, Analysis, and Governance. It refers to the process of conducting research on an issue or problem, analyzing it thoroughly, and then making decisions based on that analysis. In simpler terms, RAG is like putting together pieces of a puzzle: first, you gather information about different aspects of the issue, then analyze each piece critically, and finally, decide how to proceed with your decision. This approach is often used in business, politics, or other fields where complex problems need to


In [12]:
!nvidia-smi

Tue Oct 20 05:08:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             27W /   70W |   13757MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----